In [0]:
drop table electronics_cat.gold.curated_sales;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.curated_sales AS
SELECT
    f.order_number,
    f.line_item,

    f.order_date,
    f.year,
    f.month,

    f.customerkey,
    COALESCE(c.gender,'unknown') AS gender,
    COALESCE(c.continent,'unknown') AS continent,

    f.storekey,

    -- ✅ COUNTRY FIX
    CASE 
        WHEN f.channel = 'online' THEN 'Online'
        ELSE COALESCE(s.country,'unknown')
    END AS country,

    f.product_key,
    p.category,
    p.subcategory,

    f.quantity,
    f.unit_price_usd,
    f.exchange,

    f.revenue_usd,
    f.delivery_days,

    f.channel

FROM electronics_cat.gold.fact_sales f

LEFT JOIN electronics_cat.gold.dim_customer c
    ON f.customerkey = c.customerkey

LEFT JOIN electronics_cat.gold.dim_product p
    ON f.product_key = p.productkey

LEFT JOIN electronics_cat.gold.dim_store s
    ON f.storekey = s.store_key;

In [0]:
SELECT * FROM electronics_cat.gold.curated_sales;

In [0]:
%python
print('hi')

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.kpi1_op_revenue AS
SELECT 
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    ROUND(SUM(revenue_usd),2) AS revenue_usd
FROM electronics_cat.gold.curated_sales
WHERE YEAR(order_date) = 2020
GROUP BY YEAR(order_date), MONTH(order_date)
ORDER BY month;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.kpi2_peak_months AS
WITH yearly AS (
    SELECT SUM(revenue_usd) total
    FROM electronics_cat.gold.curated_sales
    WHERE YEAR(order_date)=2020
),
monthly AS (
    SELECT 
        MONTH(order_date) month,
        SUM(revenue_usd) revenue
    FROM electronics_cat.gold.curated_sales
    WHERE YEAR(order_date)=2020
    GROUP BY MONTH(order_date)
)
SELECT 
    m.month,
    ROUND(m.revenue,2) AS revenue_usd,
    ROUND(m.revenue*100/y.total,2) AS percent_of_total
FROM monthly m
CROSS JOIN yearly y
ORDER BY revenue_usd DESC

LIMIT 3;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.kpi3_holiday_drivers AS
WITH peak_months AS (
    SELECT month
    FROM (
        SELECT month, SUM(revenue_usd) AS rev
        FROM electronics_cat.gold.curated_sales
        WHERE year = 2020
        GROUP BY month
        ORDER BY rev DESC
        LIMIT 3
    )
)
SELECT
    category,
    ROUND(SUM(revenue_usd),2) AS peak_month_revenue,
    ROUND(
        SUM(revenue_usd)*100 / SUM(SUM(revenue_usd)) OVER(),2
    ) AS percent_of_peak_total
FROM electronics_cat.gold.curated_sales
WHERE year = 2020
  AND month IN (SELECT month FROM peak_months)
GROUP BY category
ORDER BY peak_month_revenue DESC
LIMIT 3;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.kpi4_delivery_performance AS
SELECT
    ROUND(AVG(delivery_days),2) AS avg_days,
    COUNT(*) AS total_orders
FROM electronics_cat.gold.curated_sales
WHERE delivery_days IS NOT NULL;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.kpi5_country_delivery_issues AS
SELECT
    country,
    ROUND(AVG(delivery_days),2) AS avg_days,
    COUNT(*) AS order_count,
    PERCENTILE(delivery_days,0.5) AS median_days
FROM electronics_cat.gold.curated_sales
WHERE delivery_days IS NOT NULL
GROUP BY country
ORDER BY avg_days DESC
LIMIT 5;

In [0]:

SELECT channel, COUNT(*) 
FROM electronics_cat.gold.curated_sales
GROUP BY channel;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.kpi6_channel_performance AS

SELECT
    COALESCE(continent, 'unknown') AS continent,

    
    ROUND(
        SUM(CASE WHEN channel = 'online' THEN revenue_usd ELSE 0 END) /
        NULLIF(COUNT(DISTINCT CASE WHEN channel = 'online' THEN order_number END), 0)
    ,2) AS aov_online,

    
    ROUND(
        SUM(CASE WHEN channel = 'store' THEN revenue_usd ELSE 0 END) /
        NULLIF(COUNT(DISTINCT CASE WHEN channel = 'store' THEN order_number END), 0)
    ,2) AS aov_store,

    
    COUNT(DISTINCT CASE WHEN channel = 'online' THEN order_number END) AS online_orders,
    COUNT(DISTINCT CASE WHEN channel = 'store' THEN order_number END) AS store_orders

FROM electronics_cat.gold.curated_sales

GROUP BY COALESCE(continent, 'unknown')
ORDER BY continent;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.kpi7_volume_leaders AS
SELECT
    p.category,
    SUM(s.quantity) AS total_units
FROM electronics_cat.silver.sales s
JOIN electronics_cat.silver.products p 
    ON s.product_key = p.productkey
GROUP BY p.category
ORDER BY total_units DESC
LIMIT 5;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.kpi8_revenue_leaders AS
SELECT
    p.category,
    ROUND(SUM(s.quantity * p.unit_price_usd * e.exchange), 2) AS revenue_usd
FROM electronics_cat.silver.sales s
JOIN electronics_cat.silver.products p 
    ON s.product_key = p.productkey
JOIN electronics_cat.silver.exc_rate e 
    ON s.currency_code = e.currency
    AND s.order_date = e.date
GROUP BY p.category
ORDER BY revenue_usd DESC
LIMIT 5;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.kpi9_customer_profile AS
SELECT
    c.continent,
    c.gender,
    COUNT(DISTINCT c.customerkey) AS customer_count,
    ROUND(SUM(s.quantity * p.unit_price_usd * e.exchange), 2) AS total_spent
FROM electronics_cat.silver.customers c
LEFT JOIN electronics_cat.silver.sales s 
    ON c.customerkey = s.customerkey
LEFT JOIN electronics_cat.silver.products p 
    ON s.product_key = p.productkey
LEFT JOIN electronics_cat.silver.exc_rate e 
    ON s.currency_code = e.currency
    AND s.order_date = e.date
GROUP BY c.continent, c.gender;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.kpi10_customer_loyalty AS
WITH customer_orders AS (
    SELECT
        c.customerkey,
        c.continent,
        COUNT(DISTINCT s.order_number) AS order_count
    FROM electronics_cat.silver.customers c
    LEFT JOIN electronics_cat.silver.sales s 
        ON c.customerkey = s.customerkey
    GROUP BY c.customerkey, c.continent
)
SELECT
    continent,
    ROUND(
        SUM(CASE WHEN order_count >= 2 THEN 1 ELSE 0 END) * 100.0
        / COUNT(*),
        2
    ) AS repeat_customer_rate
FROM customer_orders
GROUP BY continent;

In [0]:
SELECT * FROM electronics_cat.gold.kpi1_op_revenue;

In [0]:
SELECT * FROM electronics_cat.gold.kpi2_peak_months;

In [0]:
SELECT * FROM electronics_cat.gold.kpi3_holiday_drivers;

In [0]:
SELECT * FROM electronics_cat.gold.kpi4_delivery_performance;

In [0]:
SELECT * FROM electronics_cat.gold.kpi5_country_delivery_issues;

In [0]:
SELECT * FROM electronics_cat.gold.kpi6_channel_performance;

In [0]:
SELECT * FROM electronics_cat.gold.kpi7_volume_leaders;

In [0]:
SELECT * FROM electronics_cat.gold.kpi8_revenue_leaders;

In [0]:
SELECT * FROM electronics_cat.gold.kpi9_customer_profile;

In [0]:
SELECT * FROM electronics_cat.gold.kpi10_customer_loyalty;